# LigthGBM

Con questa quantità di dati il modello non riesce a costruire l'albero, quindi non è usabile

In [14]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Treining

In [17]:
def training(file_path, csv_name):
    # Legge il file CSV in un DataFrame pandas
    df = pd.read_csv(file_path)

    # Filtra le righe con valore valido nella colonna 'PR [SII]' (non NaN)
    df_PRvalido = df[df['PR [SII]'].notna()].copy()

    # Rimuove colonne non numeriche o non rilevanti dal DataFrame per le feature
    features = df_PRvalido.drop(columns=[
        'Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'ER [SII]', 
        'PR [SII]', 'HER2 [SII]', 'isTN', 'KI67 [%]', 'Breast'
    ])

    # Converte la colonna target in valori binari 0/1 considerando soglia 0.5
    target = (df_PRvalido['PR [SII]'] > 0.5).astype(int)

    # Crea uno scaler per normalizzare le feature (media 0 e varianza 1)
    scaler = StandardScaler()
    # Applica la normalizzazione sulle feature selezionate
    features_scaled = scaler.fit_transform(features)

    # Istanzia il modello LightGBM classificatore con random seed fisso
    model = LGBMClassifier(random_state=42)

    # Definisce una cross-validation stratificata a 5 fold (bilanciare la classe target in fold)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # Calcola gli score (accuratezza) tramite cross-validation
    scores = cross_val_score(model, features_scaled, target, cv=cv, scoring='accuracy')

    # Ritorna un dizionario con la media degli score, deviazione standard e tutti i singoli score per fold
    return {
        'mean_accuracy': scores.mean(),
        'std_accuracy': scores.std(),
        'scores_per_fold': scores
    }


# Lettura dei file

In [16]:
results = {}
print("="*50 +"\nTRAINING LightGBM\n" + "="*50)
for name, file_path in datasets.items():
    print(f"\nDataset: {name}")
    try:
        res = training(file_path, name)
        results[name] = res
        print(f"Accuracy media: {res['mean_accuracy']:.3f} ± {res['std_accuracy']:.3f}")
        print(f"Scores per fold: {[f'{s:.3f}' for s in res['scores_per_fold']]}")
    except Exception as e:
        print(f"Errore durante il training su {name}: {e}")

TRAINING LightGBM

Dataset: t2_medsam
[LightGBM] [Info] Number of positive: 28, number of negative: 25
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000211 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2084
[LightGBM] [Info] Number of data points in the train set: 53, number of used features: 109
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.528302 -> initscore=0.113329
[LightGBM] [Info] Start training from score 0.113329
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furt